# Lab 03: Central Experiment Tracking with MLflow

In this lab, you will log model training parameters, iterative loss metrics, and serialized model artifacts to our central MLflow tracking server at `http://mlflow.services.svc.cluster.local:5000`.

In [ ]:
import os
import json
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow.services.svc.cluster.local:5000")
mlflow.set_tracking_uri(tracking_uri)
print(f"🎯 MLflow Tracking URI: {tracking_uri}")

experiment_name = "starter_lab_03_tracking"
mlflow.set_experiment(experiment_name)
print(f"🧪 Active Experiment: {experiment_name}")

In [ ]:
# Train a toy model and log execution telemetry
X, y = make_classification(n_samples=100, n_features=4, random_state=42)
params = {"C": 0.5, "max_iter": 100, "solver": "lbfgs"}

with mlflow.start_run(run_name="student_starter_run") as run:
    run_id = run.info.run_id
    print(f"🚀 Logging to Run ID: {run_id}")
    
    # Log parameters
    mlflow.log_params(params)
    
    model = LogisticRegression(**params)
    model.fit(X, y)
    
    # Log metrics across iterations
    for step in range(1, 5):
        loss = 1.0 / step
        acc = 0.8 + (step * 0.04)
        mlflow.log_metric("train_loss", loss, step=step)
        mlflow.log_metric("accuracy", acc, step=step)
    
    # Log model weights & summary as file artifacts to proxied tracking server
    model_artifact_dir = "/mnt/storage/scratch/starter_model_artifact"
    os.makedirs(model_artifact_dir, exist_ok=True)
    with open(os.path.join(model_artifact_dir, "model_summary.json"), "w") as f:
        json.dump({"classes": model.classes_.tolist(), "intercept": model.intercept_.tolist(), "coef": model.coef_.tolist()}, f)
    mlflow.log_artifacts(model_artifact_dir, artifact_path="model")
    print("📦 Successfully logged parameters, metrics, and model artifact to MLflow!")

print("🎉 Lab 03 Completed Successfully!")